In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# 1. Filter PART to get the relevant keys on GPU
target_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

In [ ]:
### cell 1 ###

# 2. Early‐filter LINEITEM to just those keys
li = lineitem[lineitem.L_PARTKEY.isin(target_keys)]

# 3. Compute the 20%‐scaled average quantity per key using transform (avoids a merge)
li['avg'] = li.groupby('L_PARTKEY').L_QUANTITY.transform('mean') * 0.2

# 4. Apply the quantity filter and compute the final metric entirely on GPU
good = li[li.L_QUANTITY < li.avg]

In [ ]:
### cell 2 ###

total = pd.DataFrame({'avg_yearly': [good.L_EXTENDEDPRICE.sum() / 7.0]})